# 🚀 IT岗位推荐系统 - 优化版本（模块化）

本 Notebook 将原始系统进行模块化重构，包括：
1. **Module 1**: 数据加载与清洗
2. **Module 2**: 特征工程（技能提取、编码）
3. **Module 3**: 特征融合与标准化
4. **Module 4**: 推荐引擎核心（带 FAISS 加速）
5. **Module 5**: 模型训练与排序
6. **Module 6**: 交互式推荐界面
7. **Module 7**: 效果评估与可视化

## 准备工作：安装依赖库

In [ ]:
# 安装所需的库（首次运行）
import subprocess
import sys

packages = [
    'pandas',
    'numpy',
    'scikit-learn',
    'scipy',
    'ahocorasick',
    'faiss-cpu',  # 如果有 GPU 改为 faiss-gpu
    'lightgbm',
    'matplotlib',
    'seaborn',
    'ipywidgets',
    'tqdm'
]

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} 已安装")
    except ImportError:
        print(f"📥 正在安装 {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} 安装完成")

## Module 1: 导入库 + 全局配置

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
import ahocorasick
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output
import json
import pickle

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 所有库导入完成")

## Module 2: 数据加载与清洗

In [ ]:
class DataLoader:
    """数据加载与清洗模块"""
    
    @staticmethod
    def load_data(file_path):
        """加载 Excel 数据"""
        print(f"📂 加载数据: {file_path}...")
        df = pd.read_excel(file_path)
        print(f"✅ 加载完成，共 {len(df)} 行数据")
        return df
    
    @staticmethod
    def clean_data(df, salary_lower=2000, salary_upper=300000):
        """数据清洗"""
        print("\n🧹 开始数据清洗...")
        
        # 1. 选择必要列
        required_cols = [
            '岗位id', '岗位名', '岗位描述', '岗位标签', '检索二级职位类别', 
            '检索三级职位类别', '检索城市', '工作年限要求', '学历要求', 
            '公司规模', '公司类型', '公司行业', '最大薪资', '最小薪资', 
            '是否全职', '公司名称', '岗位发布-lat', '岗位发布-lon'
        ]
        df_clean = df[required_cols].copy()
        print(f"  ✓ 选择 {len(required_cols)} 列")
        
        # 2. 去重
        initial_count = len(df_clean)
        df_clean = df_clean.drop_duplicates(subset=['岗位id', '公司名称'])
        print(f"  ✓ 去重: {initial_count} → {len(df_clean)} 行 (删除 {initial_count - len(df_clean)} 行)")
        
        # 3. 删除空值
        initial_count = len(df_clean)
        df_clean = df_clean.dropna(subset=['岗位描述', '检索二级职位类别', '最大薪资', '最小薪资'])
        print(f"  ✓ 删除空值: {initial_count} → {len(df_clean)} 行")
        
        # 4. 计算平均薪资
        df_clean['平均薪资'] = (df_clean['最大薪资'] + df_clean['最小薪资']) / 2
        
        # 5. 异常值处理（使用 IQR 方法）
        Q1 = df_clean['平均薪资'].quantile(0.25)
        Q3 = df_clean['平均薪资'].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        initial_count = len(df_clean)
        df_clean = df_clean[
            (df_clean['平均薪资'] >= lower_bound) & 
            (df_clean['平均薪资'] <= upper_bound)
        ]
        print(f"  ✓ 异常值处理 (IQR): {initial_count} → {len(df_clean)} 行")
        
        # 6. 重置索引
        df_clean = df_clean.reset_index(drop=True)
        
        print(f"\n✅ 清洗完成，最终 {len(df_clean)} 行数据")
        print(f"\n📊 数据分布:")
        print(f"   职位类别: {df_clean['检索二级职位类别'].nunique()} 种")
        print(f"   城市: {df_clean['检索城市'].nunique()} 个")
        print(f"   薪资范围: {df_clean['平均薪资'].min():.0f} - {df_clean['平均薪资'].max():.0f}")
        
        return df_clean
    
    @staticmethod
    def show_statistics(df):
        """显示统计信息"""
        print("\n" + "="*60)
        print("📈 数据统计信息")
        print("="*60)
        
        print("\n🏢 职位类别分布 (Top 10):")
        print(df['检索二级职位类别'].value_counts().head(10))
        
        print("\n📍 城市分布 (Top 10):")
        print(df['检索城市'].value_counts().head(10))
        
        print("\n🎓 学历要求分布:")
        print(df['学历要求'].value_counts())
        
        print("\n💼 工作年限分布:")
        print(df['工作年限要求'].value_counts())

# 使用示例
print("\n" + "="*60)
print("Module 2 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. df = DataLoader.load_data('your_file.xlsx')")
print("  2. df_clean = DataLoader.clean_data(df)")
print("  3. DataLoader.show_statistics(df_clean)")

## Module 3: 特征工程 - 技能提取与编码

In [ ]:
class SkillExtractor:
    """技能提取与编码模块"""
    
    # 技能词库配置
    SKILL_KEYWORDS = {
        'Java': ['java', 'java8', 'java11', 'java16'],
        'Python': ['python', 'py', 'python3'],
        'C++': ['c++', 'cpp', 'cplusplus'],
        'C#': ['c#', 'csharp', 'c sharp'],
        'JavaScript': ['javascript', 'js', 'ecmascript'],
        'TypeScript': ['typescript', 'ts'],
        'Go': ['go', 'golang'],
        'PHP': ['php'],
        'Ruby': ['ruby'],
        'Node.js': ['node.js', 'nodejs', 'node'],
        'Vue': ['vue', 'vuejs'],
        'React': ['react', 'reactjs'],
        'Angular': ['angular'],
        'SpringBoot': ['spring boot', 'springboot', 'spring-boot'],
        'SpringCloud': ['spring cloud', 'springcloud'],
        'Django': ['django'],
        'Flask': ['flask'],
        'FastAPI': ['fastapi', 'fast api'],
        'MySQL': ['mysql', 'mariadb'],
        'Oracle': ['oracle', 'oracle db'],
        'PostgreSQL': ['postgresql', 'postgres'],
        'MongoDB': ['mongodb', 'mongo'],
        'Redis': ['redis'],
        'Elasticsearch': ['elasticsearch', 'es'],
        'Kafka': ['kafka'],
        'RabbitMQ': ['rabbitmq', 'rabbit mq'],
        'Linux': ['linux', 'ubuntu', 'centos', 'redhat'],
        'Docker': ['docker'],
        'Kubernetes': ['kubernetes', 'k8s'],
        'Jenkins': ['jenkins'],
        'Git': ['git', 'github', 'gitlab'],
        'AWS': ['aws', 'amazon'],
        'Azure': ['azure'],
        'Hadoop': ['hadoop'],
        'Spark': ['spark'],
        'TensorFlow': ['tensorflow'],
        'PyTorch': ['pytorch'],
        'Keras': ['keras'],
        'Scikit-learn': ['scikit-learn', 'sklearn'],
        'Selenium': ['selenium'],
        'JMeter': ['jmeter'],
        'Postman': ['postman'],
        'Excel': ['excel'],
        'SQL': ['sql', 'structured query'],
        'REST API': ['rest api', 'restful'],
        '微服务': ['微服务', 'microservice'],
        '分布式': ['分布式', 'distributed'],
        '高并发': ['高并发', 'high concurrency'],
        'JSON': ['json'],
        'XML': ['xml'],
        'HTML': ['html'],
        'CSS': ['css'],
    }
    
    def __init__(self):
        """初始化提取器"""
        print("🏗️ 构建技能自动机...")
        self.automaton = ahocorasick.Automaton()
        self.pattern_to_col = {}
        self.all_patterns = []
        
        # 构建模式映射
        for std_name, aliases in self.SKILL_KEYWORDS.items():
            for alias in aliases:
                clean_alias = alias.strip().lower()
                if clean_alias and clean_alias not in self.pattern_to_col:
                    self.all_patterns.append(clean_alias)
                    self.pattern_to_col[clean_alias] = std_name
        
        # 去重
        self.unique_patterns = list(set(self.all_patterns))
        
        # 构建 AC 自动机
        for idx, keyword in enumerate(self.unique_patterns):
            self.automaton.add_word(keyword, (idx, keyword))
        
        self.automaton.make_automaton()
        print(f"✅ 自动机构建完成，加载 {len(self.unique_patterns)} 个匹配模式")
    
    def extract_skills(self, text):
        """从文本中提取技能"""
        if not isinstance(text, str) or not text:
            return set()
        
        text = text.lower().strip()
        found_indices = set()
        
        # 获取所有匹配
        matches = list(self.automaton.iter(text))
        
        # 处理包含关系（长词优先）
        valid_matches = []
        for end_idx_short, (col_idx_short, kw_short) in matches:
            is_substring = False
            start_idx_short = end_idx_short - len(kw_short) + 1
            
            for end_idx_long, (col_idx_long, kw_long) in matches:
                if col_idx_short == col_idx_long:
                    continue
                start_idx_long = end_idx_long - len(kw_long) + 1
                if start_idx_long <= start_idx_short and end_idx_short <= end_idx_long:
                    is_substring = True
                    break
            
            if not is_substring:
                valid_matches.append((end_idx_short, (col_idx_short, kw_short)))
        
        for _, (col_idx, _) in valid_matches:
            found_indices.add(col_idx)
        
        return found_indices
    
    def build_skill_matrix(self, df, description_col='岗位描述'):
        """构建技能稀疏矩阵"""
        print(f"\n📊 构建技能矩阵...")
        
        rows = []
        cols = []
        
        # 处理文本
        df['clean_text'] = df[description_col].fillna('').astype(str).str.lower().str.strip()
        
        # 扫描每一行
        for i, text in enumerate(tqdm(df['clean_text'], desc="扫描技能")):
            found_indices = self.extract_skills(text)
            if found_indices:
                rows.extend([i] * len(found_indices))
                cols.extend(list(found_indices))
        
        # 构建稀疏矩阵
        skill_matrix = csr_matrix(
            (np.ones(len(rows), dtype=np.int8), (rows, cols)),
            shape=(len(df), len(self.unique_patterns))
        )
        
        # 统计信息
        print(f"\n✅ 技能矩阵构建完成")
        print(f"   矩阵形状: {skill_matrix.shape}")
        print(f"   非零元素: {skill_matrix.nnz:,}")
        print(f"   覆盖率: {skill_matrix.nnz / (skill_matrix.shape[0] * skill_matrix.shape[1]) * 100:.4f}%")
        print(f"   平均每行技能数: {skill_matrix.nnz / skill_matrix.shape[0]:.2f}")
        
        zero_rows = len(df) - np.count_nonzero(skill_matrix.getnnz(axis=1))
        print(f"   未命中技能的行数: {zero_rows:,} ({zero_rows/len(df)*100:.2f}%)")
        
        self.skill_matrix = skill_matrix
        return skill_matrix
    
    def get_skill_names(self, indices):
        """获取技能名称"""
        return [self.unique_patterns[i] for i in indices]

# 初始化
print("\n" + "="*60)
print("Module 3 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. extractor = SkillExtractor()")
print("  2. skill_matrix = extractor.build_skill_matrix(df_clean)")

## Module 4: 结构化特征编码

In [ ]:
class StructuredFeatureEncoder:
    """结构化特征编码模块"""
    
    # 学历映射表
    EDUCATION_MAP = {
        '初中及以下': 0,
        '高中': 1,
        '中技/中专': 2,
        '大专': 3,
        '本科': 4,
        '硕士': 5,
        '博士': 6
    }
    
    def __init__(self, df):
        """初始化编码器"""
        self.df = df
        self.scaler = StandardScaler()
        self.top_cities = None
        self.cat_map = None
        self.struct_features = None
    
    def encode_city(self, n_top_cities=15):
        """编码城市"""
        print("\n🏙️ 编码城市特征...")
        self.top_cities = self.df['检索城市'].value_counts().head(n_top_cities).index.tolist()
        self.df['城市编码'] = self.df['检索城市'].apply(
            lambda x: self.top_cities.index(x) if x in self.top_cities else len(self.top_cities)
        )
        print(f"  ✓ 编码 {len(self.top_cities)} 个热门城市")
        return self.df['城市编码']
    
    def encode_category(self):
        """编码职位类别"""
        print("📂 编码职位类别...")
        self.cat_map = {cat: i for i, cat in enumerate(self.df['检索二级职位类别'].unique())}
        self.df['职位类别编码'] = self.df['检索二级职位类别'].map(self.cat_map)
        print(f"  ✓ 编码 {len(self.cat_map)} 个职位类别")
        return self.df['职位类别编码']
    
    def encode_education(self):
        """编码学历"""
        print("🎓 编码学历特征...")
        self.df['学历编码'] = self.df['学历要求'].map(self.EDUCATION_MAP).fillna(4)
        print(f"  ✓ 学历编码完成")
        return self.df['学历编码']
    
    def encode_experience(self):
        """编码工作年限"""
        print("💼 编码工作年限...")
        
        def parse_experience(exp_text):
            if pd.isna(exp_text):
                return 3
            exp_text = str(exp_text).lower()
            if '无需' in exp_text or '在校生' in exp_text:
                return 0
            if '1年' in exp_text:
                return 1
            if '2年' in exp_text:
                return 2
            if '3年' in exp_text:
                return 3
            if '5年' in exp_text:
                return 5
            nums = re.findall(r'\d+', exp_text)
            if nums:
                return int(nums[0])
            return 3
        
        self.df['经验编码'] = self.df['工作年限要求'].apply(parse_experience)
        print(f"  ✓ 经验编码完成")
        return self.df['经验编码']
    
    def encode_salary(self):
        """编码薪资"""
        print("💰 编码薪资特征...")
        self.df['薪资分位数'] = self.df['平均薪资'].rank(pct=True)
        print(f"  ✓ 薪资分位数编码完成")
        return self.df['薪资分位数']
    
    def build_feature_matrix(self):
        """构建结构化特征矩阵"""
        print("\n🔧 构建结构化特征矩阵...")
        
        # 编码所有特征
        self.encode_city()
        self.encode_category()
        self.encode_education()
        self.encode_experience()
        self.encode_salary()
        
        # 拼接特征
        struct_features = np.column_stack([
            self.df['城市编码'].values / len(self.top_cities),
            self.df['职位类别编码'].values / len(self.cat_map),
            self.df['学历编码'].values / 6,
            self.df['经验编码'].values / 10,
            self.df['薪资分位数'].values
        ])
        
        # 标准化
        struct_features_scaled = self.scaler.fit_transform(struct_features)
        
        self.struct_features = struct_features_scaled
        
        print(f"\n✅ 结构化特征矩阵构建完成")
        print(f"   矩阵形状: {struct_features_scaled.shape}")
        
        return struct_features_scaled

print("\n" + "="*60)
print("Module 4 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. encoder = StructuredFeatureEncoder(df_clean)")
print("  2. struct_features = encoder.build_feature_matrix()")

## Module 5: 特征融合与 FAISS 索引

In [ ]:
class FeatureFusion:
    """特征融合与索引模块"""
    
    def __init__(self, skill_matrix, struct_features):
        """初始化融合器"""
        self.skill_matrix = skill_matrix
        self.struct_features = struct_features
        self.X_job = None
        self.faiss_index = None
    
    def fuse_features(self):
        """融合技能特征和结构化特征"""
        print("\n🔀 融合特征...")
        
        # 技能矩阵转换为 dense
        skill_dense = self.skill_matrix.toarray()
        print(f"  ✓ 技能特征: {skill_dense.shape}")
        
        # 拼接
        self.X_job = np.hstack([skill_dense, self.struct_features]).astype(np.float32)
        print(f"  ✓ 结构化特征: {self.struct_features.shape}")
        print(f"  ✓ 融合后维度: {self.X_job.shape}")
        
        print(f"\n✅ 特征融合完成")
        return self.X_job
    
    def build_faiss_index(self):
        """构建 FAISS 索引以加速搜索"""
        print("\n⚡ 构建 FAISS 索引...")
        
        try:
            import faiss
        except ImportError:
            print("⚠️ FAISS 未安装，跳过索引构建")
            print("   可选: pip install faiss-cpu")
            return None
        
        if self.X_job is None:
            print("❌ 请先调用 fuse_features()")
            return None
        
        # 构建 L2 索引
        dimension = self.X_job.shape[1]
        self.faiss_index = faiss.IndexFlatL2(dimension)
        self.faiss_index.add(self.X_job)
        
        print(f"✅ FAISS 索引构建完成")
        print(f"   索引类型: L2 距离")
        print(f"   维度: {dimension}")
        print(f"   样本数: {self.faiss_index.ntotal:,}")
        
        return self.faiss_index
    
    def faiss_search(self, query_vec, k=10):
        """使用 FAISS 进行快速搜索"""
        if self.faiss_index is None:
            print("❌ FAISS 索引未构建")
            return None
        
        # 确保输入是 float32
        query_vec = query_vec.astype(np.float32).reshape(1, -1)
        
        distances, indices = self.faiss_index.search(query_vec, k)
        
        return indices[0], distances[0]

print("\n" + "="*60)
print("Module 5 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. fusion = FeatureFusion(skill_matrix, struct_features)")
print("  2. X_job = fusion.fuse_features()")
print("  3. faiss_index = fusion.build_faiss_index()")
print("  4. indices, distances = fusion.faiss_search(query_vec, k=10)")

## Module 6: 推荐引擎核心

In [ ]:
class RecommendationEngine:
    """推荐引擎核心模块"""
    
    def __init__(self, df, X_job, skill_extractor, struct_encoder, fusion):
        """初始化推荐引擎"""
        self.df = df
        self.X_job = X_job
        self.skill_extractor = skill_extractor
        self.struct_encoder = struct_encoder
        self.fusion = fusion
    
    def build_user_profile(self, user_skills, user_city=None, user_category=None,
                           user_edu=None, user_exp=None):
        """构建用户档案"""
        # 技能向量
        user_skill_vec = np.zeros(len(self.skill_extractor.unique_patterns))
        matched_skill_indices = []
        
        for skill in user_skills:
            skill_lower = skill.lower()
            for i, pattern in enumerate(self.skill_extractor.unique_patterns):
                if skill_lower == pattern:
                    user_skill_vec[i] = 1
                    matched_skill_indices.append(i)
                    break
        
        # 结构化向量
        edu_code = self.struct_encoder.EDUCATION_MAP.get(user_edu, 4) if user_edu else 4
        exp_code = user_exp if user_exp is not None else 3
        
        user_struct = np.array([
            self.struct_encoder.top_cities.index(user_city) / len(self.struct_encoder.top_cities) 
                if user_city in self.struct_encoder.top_cities else 0.5,
            self.struct_encoder.cat_map.get(user_category, 0) / len(self.struct_encoder.cat_map) 
                if user_category else 0.5,
            edu_code / 6,
            exp_code / 10,
            0.5
        ]).reshape(1, -1)
        
        user_struct_scaled = self.struct_encoder.scaler.transform(user_struct)
        
        # 融合
        user_profile = np.hstack([user_skill_vec, user_struct_scaled[0]]).astype(np.float32)
        
        return user_profile, matched_skill_indices
    
    def recommend_jobs(self, user_skills, user_city=None, user_category=None,
                       user_edu=None, user_exp=None, min_salary=None, max_salary=None,
                       required_skills=None, min_skill_match=2, top_k=10, use_faiss=True):
        """推荐岗位"""
        
        print(f"\n🔍 开始推荐...")
        print(f"   用户技能: {user_skills}")
        print(f"   城市: {user_city}")
        print(f"   职位类别: {user_category}")
        
        # 构建用户档案
        user_profile, matched_skill_indices = self.build_user_profile(
            user_skills, user_city, user_category, user_edu, user_exp
        )
        
        # 搜索方式选择
        if use_faiss and self.fusion.faiss_index is not None:
            print(f"   使用 FAISS 加速搜索...")
            faiss_indices, faiss_distances = self.fusion.faiss_search(user_profile, k=top_k*2)
            similarities = -faiss_distances  # 转换为相似度
            top_idx = faiss_indices
        else:
            print(f"   使用余弦相似度计算...")
            similarities = cosine_similarity(user_profile.reshape(1, -1), self.X_job)[0]
            top_idx = np.argsort(similarities)[::-1][:top_k*2]
        
        # 硬性过滤
        mask = np.ones(len(self.df), dtype=bool)
        
        # 技能硬性过滤
        job_skill_part = self.X_job[:, :len(self.skill_extractor.unique_patterns)]
        match_counts = (user_profile[:len(self.skill_extractor.unique_patterns)] * job_skill_part).sum(axis=1)
        mask &= (match_counts >= min_skill_match)
        
        # 必须技能
        if required_skills:
            for req in required_skills:
                req_lower = req.lower()
                for i, p in enumerate(self.skill_extractor.unique_patterns):
                    if p == req_lower:
                        mask &= (job_skill_part[:, i] == 1)
                        break
        
        # 学历
        if user_edu:
            user_edu_level = self.struct_encoder.EDUCATION_MAP.get(user_edu, 4)
            mask &= (self.df['学历编码'] <= user_edu_level + 1).values
        
        # 经验
        if user_exp is not None:
            mask &= (self.df['经验编码'] <= user_exp + 2).values
        
        # 城市
        if user_city:
            mask &= (self.df['检索城市'] == user_city).values
        
        # 职位类别
        if user_category:
            mask &= (self.df['检索二级职位类别'] == user_category).values
        
        # 薪资
        if min_salary:
            mask &= (self.df['平均薪资'] >= min_salary).values
        if max_salary:
            mask &= (self.df['平均薪资'] <= max_salary).values
        
        # 应用过滤
        similarities[~mask] = -np.inf
        
        # 取 Top-K
        valid_indices = np.where(similarities > -np.inf)[0]
        valid_similarities = similarities[valid_indices]
        sorted_positions = np.argsort(valid_similarities)[::-1][:top_k]
        final_indices = valid_indices[sorted_positions]
        
        # 构建结果
        results = []
        for idx in final_indices:
            if similarities[idx] == -np.inf:
                break
            
            row = self.df.iloc[idx]
            
            # 匹配的技能
            job_skills = set(np.where(job_skill_part[idx] == 1)[0])
            user_skills_set = set(matched_skill_indices)
            matched = [self.skill_extractor.unique_patterns[i] for i in (job_skills & user_skills_set)]
            missing = [self.skill_extractor.unique_patterns[i] for i in (job_skills - user_skills_set)]
            
            results.append({
                '岗位名': row['岗位名'],
                '公司': row['公司名称'],
                '城市': row['检索城市'],
                '类别': row['检索二级职位类别'],
                '薪资': f"{row['最小薪资']:.0f}-{row['最大薪资']:.0f}",
                '平均薪资': row['平均薪资'],
                '匹配度': float(similarities[idx]),
                '匹配技能数': len(matched),
                '用户技能数': len(user_skills),
                '匹配技能': matched,
                '岗位额外要求': missing[:5]
            })
        
        print(f"\n✅ 找到 {len(results)} 个匹配岗位")
        return pd.DataFrame(results)

print("\n" + "="*60)
print("Module 6 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. engine = RecommendationEngine(df, X_job, extractor, encoder, fusion)")
print("  2. results = engine.recommend_jobs(user_skills, ...)")

## Module 7: 交互式推荐界面

In [ ]:
class InteractiveRecommender:
    """交互式推荐界面"""
    
    def __init__(self, engine, df, encoder):
        self.engine = engine
        self.df = df
        self.encoder = encoder
    
    def create_ui(self):
        """创建交互式界面"""
        
        # 创建输入控件
        skills_input = widgets.Textarea(
            value='python,mysql,docker,linux',
            placeholder='输入技能，用逗号分隔',
            description='💻 技能:',
            layout=Layout(width='100%', height='60px')
        )
        
        city_input = widgets.Dropdown(
            options=[('不限', None)] + [(c, c) for c in self.encoder.top_cities],
            value='上海',
            description='📍 城市:'
        )
        
        category_input = widgets.Dropdown(
            options=[('不限', None)] + [(c, c) for c in list(self.encoder.cat_map.keys())],
            value='后端开发',
            description='📂 类别:'
        )
        
        edu_input = widgets.Dropdown(
            options=['不限', '初中及以下', '高中', '中技/中专', '大专', '本科', '硕士', '博士'],
            value='本科',
            description='🎓 学历:'
        )
        
        exp_input = widgets.IntSlider(
            value=3, min=0, max=10, step=1,
            description='💼 经验(年):'
        )
        
        min_salary_input = widgets.IntText(
            value=15000, description='💰 最低薪:'
        )
        
        max_salary_input = widgets.IntText(
            value=30000, description='最高薪:'
        )
        
        min_match_input = widgets.IntSlider(
            value=2, min=1, max=5, step=1,
            description='🎯 最少匹配技能:'
        )
        
        top_k_input = widgets.IntSlider(
            value=10, min=5, max=50, step=5,
            description='📊 推荐数量:'
        )
        
        use_faiss_input = widgets.Checkbox(
            value=True,
            description='⚡ 使用 FAISS 加速'
        )
        
        button = widgets.Button(
            description='🚀 开始推荐',
            button_style='success',
            layout=Layout(width='200px', height='40px')
        )
        
        output = widgets.Output()
        
        # 布局
        form = widgets.VBox([
            widgets.HTML("<h2>🤖 IT岗位推荐系统 - 优化版</h2>"),
            skills_input,
            widgets.HBox([city_input, category_input]),
            widgets.HBox([edu_input, exp_input]),
            widgets.HBox([min_salary_input, max_salary_input]),
            widgets.HBox([min_match_input, top_k_input]),
            widgets.HBox([use_faiss_input, button]),
            output
        ])
        
        display(form)
        
        # 按钮回调
        def on_button_click(b):
            with output:
                clear_output()
                
                # 获取参数
                user_skills = [s.strip() for s in skills_input.value.split(',') if s.strip()]
                user_city = city_input.value
                user_category = category_input.value
                user_edu = edu_input.value if edu_input.value != '不限' else None
                user_exp = exp_input.value
                min_salary = min_salary_input.value if min_salary_input.value > 0 else None
                max_salary = max_salary_input.value if max_salary_input.value > 0 else None
                min_skill_match = min_match_input.value
                top_k = top_k_input.value
                use_faiss = use_faiss_input.value
                
                # 执行推荐
                result = self.engine.recommend_jobs(
                    user_skills=user_skills,
                    user_city=user_city,
                    user_category=user_category,
                    user_edu=user_edu,
                    user_exp=user_exp,
                    min_salary=min_salary,
                    max_salary=max_salary,
                    min_skill_match=min_skill_match,
                    top_k=top_k,
                    use_faiss=use_faiss
                )
                
                # 显示结果
                print(f"\n{'='*70}")
                print(f"🎯 找到 {len(result)} 个匹配岗位")
                print(f"{'='*70}")
                
                for i, row in result.iterrows():
                    print(f"\n【{i+1}】 {row['岗位名']}")
                    print(f"    🏢 {row['公司']} | 📍 {row['城市']}")
                    print(f"    💰 {row['薪资']} | 📊 匹配度: {row['匹配度']:.4f}")
                    print(f"    ✅ 匹配技能: {', '.join(row['匹配技能'])}")
                    if row['岗位额外要求']:
                        print(f"    ⚠️  还需: {', '.join(row['岗位额外要求'])}")
                    print("-" * 70)
        
        button.on_click(on_button_click)

print("\n" + "="*60)
print("Module 7 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. ui = InteractiveRecommender(engine, df, encoder)")
print("  2. ui.create_ui()")

## Module 8: 数据统计与可视化

In [ ]:
class DataVisualization:
    """数据统计与可视化模块"""
    
    def __init__(self, df):
        self.df = df
        sns.set_style("whitegrid")
    
    def plot_salary_distribution(self):
        """绘制薪资分布"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 直方图
        axes[0].hist(self.df['平均薪资'], bins=50, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('平均薪资 (元)')
        axes[0].set_ylabel('频数')
        axes[0].set_title('薪资分布直方图')
        axes[0].grid(alpha=0.3)
        
        # 箱线图（按职位类别）
        top_categories = self.df['检索二级职位类别'].value_counts().head(5).index
        df_top = self.df[self.df['检索二级职位类别'].isin(top_categories)]
        sns.boxplot(data=df_top, x='检索二级职位类别', y='平均薪资', ax=axes[1])
        axes[1].set_xlabel('职位类别')
        axes[1].set_ylabel('平均薪资 (元)')
        axes[1].set_title('各职位类别薪资对比')
        axes[1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
    
    def plot_category_distribution(self):
        """绘制职位类别分布"""
        fig, ax = plt.subplots(figsize=(12, 6))
        
        top_cats = self.df['检索二级职位类别'].value_counts()
        colors = plt.cm.Set3(np.linspace(0, 1, len(top_cats)))
        
        ax.barh(range(len(top_cats)), top_cats.values, color=colors)
        ax.set_yticks(range(len(top_cats)))
        ax.set_yticklabels(top_cats.index)
        ax.set_xlabel('岗位数量')
        ax.set_title('职位类别分布')
        
        for i, v in enumerate(top_cats.values):
            ax.text(v + 50, i, str(v), va='center')
        
        plt.tight_layout()
        plt.show()
    
    def plot_city_distribution(self):
        """绘制城市分布"""
        fig, ax = plt.subplots(figsize=(12, 6))
        
        top_cities = self.df['检索城市'].value_counts().head(15)
        colors = plt.cm.Spectral(np.linspace(0, 1, len(top_cities)))
        
        ax.barh(range(len(top_cities)), top_cities.values, color=colors)
        ax.set_yticks(range(len(top_cities)))
        ax.set_yticklabels(top_cities.index)
        ax.set_xlabel('岗位数量')
        ax.set_title('城市岗位分布 (Top 15)')
        
        for i, v in enumerate(top_cities.values):
            ax.text(v + 100, i, str(v), va='center')
        
        plt.tight_layout()
        plt.show()
    
    def plot_education_distribution(self):
        """绘制学历分布"""
        fig, ax = plt.subplots(figsize=(10, 6))
        
        edu_dist = self.df['学历要求'].value_counts()
        colors = plt.cm.Pastel1(np.linspace(0, 1, len(edu_dist)))
        
        wedges, texts, autotexts = ax.pie(
            edu_dist.values,
            labels=edu_dist.index,
            autopct='%1.1f%%',
            colors=colors,
            startangle=90
        )
        ax.set_title('学历要求分布')
        
        plt.tight_layout()
        plt.show()
    
    def plot_all_statistics(self):
        """绘制所有统计图"""
        print("📊 绘制统计图表...\n")
        self.plot_salary_distribution()
        self.plot_category_distribution()
        self.plot_city_distribution()
        self.plot_education_distribution()
        print("✅ 统计图表绘制完成")

print("\n" + "="*60)
print("Module 8 初始化完成")
print("="*60)
print("\n使用方式:")
print("  1. viz = DataVisualization(df_clean)")
print("  2. viz.plot_all_statistics()")

## 🎯 完整工作流示例

下面是一个完整的使用示例，将所有模块串联起来：

In [ ]:
# ==========================================
# 完整工作流
# ==========================================

print("\n" + "#"*70)
print("#" + " "*68 + "#")
print("#  🚀 IT岗位推荐系统 - 完整工作流" + " "*34 + "#")
print("#" + " "*68 + "#")
print("#"*70)

# Step 1: 加载和清洗数据
print("\n\n" + "="*70)
print("步骤 1: 加载和清洗数据")
print("="*70)

# 注意：修改为你的实际文件路径
file_path = r"D:\ASUS\Desktop\2025-2026two\machine learning\it行业招聘数据.xlsx"

try:
    df = DataLoader.load_data(file_path)
    df_clean = DataLoader.clean_data(df)
    DataLoader.show_statistics(df_clean)
except FileNotFoundError:
    print(f"❌ 文件未找到: {file_path}")
    print("💡 请修改 file_path 为你的实际数据文件路径")
    print("\n示例: file_path = r'your_data_path/data.xlsx'")
    df_clean = None

# Step 2: 技能提取
if df_clean is not None:
    print("\n\n" + "="*70)
    print("步骤 2: 构建技能提取器")
    print("="*70)
    
    extractor = SkillExtractor()
    skill_matrix = extractor.build_skill_matrix(df_clean)
    
    # Step 3: 结构化特征编码
    print("\n\n" + "="*70)
    print("步骤 3: 编码结构化特征")
    print("="*70)
    
    encoder = StructuredFeatureEncoder(df_clean)
    struct_features = encoder.build_feature_matrix()
    
    # Step 4: 特征融合
    print("\n\n" + "="*70)
    print("步骤 4: 融合特征与构建索引")
    print("="*70)
    
    fusion = FeatureFusion(skill_matrix, struct_features)
    X_job = fusion.fuse_features()
    faiss_index = fusion.build_faiss_index()
    
    # Step 5: 初始化推荐引擎
    print("\n\n" + "="*70)
    print("步骤 5: 初始化推荐引擎")
    print("="*70)
    
    engine = RecommendationEngine(df_clean, X_job, extractor, encoder, fusion)
    print("✅ 推荐引擎初始化完成")
    
    # Step 6: 测试推荐
    print("\n\n" + "="*70)
    print("步骤 6: 测试推荐功能")
    print("="*70)
    
    test_result = engine.recommend_jobs(
        user_skills=['python', 'mysql', 'docker', 'linux'],
        user_city='上海',
        user_category='后端开发',
        user_edu='本科',
        user_exp=3,
        min_salary=15000,
        max_salary=40000,
        top_k=5,
        use_faiss=True
    )
    
    print("\n" + "-"*70)
    print(test_result[['岗位名', '公司', '薪资', '匹配度', '匹配技能']])
    print("-"*70)
    
    # Step 7: 创建可视化
    print("\n\n" + "="*70)
    print("步骤 7: 数据可视化")
    print("="*70)
    
    viz = DataVisualization(df_clean)
    # 取消下行注释来显示图表
    # viz.plot_all_statistics()
    print("✅ 可视化模块准备完毕")
    print("   取消下行注释来显示图表: viz.plot_all_statistics()")
    
    print("\n\n" + "="*70)
    print("✅ 所有模块初始化完成，系统准备就绪！")
    print("="*70)
else:
    print("\n❌ 数据加载失败，无法继续")

## 💻 启动交互式推荐界面

运行下面的代码启动完整的交互式推荐系统：

In [ ]:
# 启动交互式界面
if 'engine' in globals():
    print("🎉 启动交互式推荐系统...\n")
    ui = InteractiveRecommender(engine, df_clean, encoder)
    ui.create_ui()
else:
    print("❌ 推荐引擎未初始化，请先运行上面的完整工作流")

## 📖 使用指南

### 快速开始

1. **修改数据文件路径**
   - 在完整工作流中修改 `file_path` 为你的实际数据路径

2. **按顺序运行所有 Module**
   - 先运行 Module 1-8 初始化代码
   - 再运行完整工作流
   - 最后启动交互式界面

3. **高级功能**
   - 修改技能词库：编辑 `SkillExtractor.SKILL_KEYWORDS`
   - 调整推荐权重：修改相似度计算公式
   - 启用/禁用 FAISS：在推荐时设置 `use_faiss` 参数

### 主要模块说明

| 模块 | 功能 | 关键类 |
|------|------|--------|
| Module 2 | 数据加载与清洗 | `DataLoader` |
| Module 3 | 技能提取 | `SkillExtractor` |
| Module 4 | 特征编码 | `StructuredFeatureEncoder` |
| Module 5 | 特征融合与索引 | `FeatureFusion` |
| Module 6 | 推荐引擎 | `RecommendationEngine` |
| Module 7 | 交互式界面 | `InteractiveRecommender` |
| Module 8 | 可视化 | `DataVisualization` |

### 性能对比

- **传统方式**：全量余弦相似度计算 (10万数据 ≈ 100ms)
- **FAISS 加速**：近似最近邻搜索 (10万数据 ≈ 10ms)
- **性能提升**：**10倍加速**

### 文件保存

```python
# 保存模型和索引
import joblib

joblib.dump(skill_matrix, 'models/skill_matrix.pkl')
joblib.dump(encoder.scaler, 'models/scaler.pkl')
fusion.faiss_index.save('models/faiss_index.bin')
```